In [50]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
import matplotlib.pyplot as plt
from datetime import datetime

In [51]:
# Ignore warnings for cleaner output
warnings.filterwarnings("ignore")

In [53]:
def prepare_time_series_by_drug(df):
    # Convert Date to datetime
    df['Date'] = pd.to_datetime(df['Date'])

    # Get unique drug names
    drug_names = df['Drug Name'].unique()

    # Create dictionary to store time series for each drug
    drug_series = {}

    for drug in drug_names:
        # Filter data for this drug
        drug_data = df[df['Drug Name'] == drug]
        # Group by date and sum sales
        daily_sales = drug_data.groupby('Date')['Sales'].sum().reset_index()
        daily_sales.set_index('Date', inplace=True)
        drug_series[drug] = daily_sales

    return drug_series

In [54]:
def find_best_sarima_params(data):
    """
    Grid search for SARIMA parameters
    Returns the parameters with lowest AIC
    """
    best_aic = float('inf')
    best_params = None

    # Define parameter grid - simplified for faster processing
    p_values = range(0, 2)
    d_values = range(0, 2)
    q_values = range(0, 2)
    P_values = range(0, 2)
    D_values = range(0, 2)
    Q_values = range(0, 2)
    s = 12  # Monthly seasonality

    for p in p_values:
        for d in d_values:
            for q in q_values:
                for P in P_values:
                    for D in D_values:
                        for Q in Q_values:
                            try:
                                model = SARIMAX(
                                    data,
                                    order=(p, d, q),
                                    seasonal_order=(P, D, Q, s),
                                    enforce_stationarity=False,
                                    enforce_invertibility=False
                                )
                                results = model.fit(disp=False)

                                if results.aic < best_aic:
                                    best_aic = results.aic
                                    best_params = (p, d, q, P, D, Q)

                            except:
                                continue

    return best_params

This approach is known as SARIMA Hyperparameter Tuning using Grid Search.
It's a brute-force method that evaluates multiple SARIMA models and selects the one with the best AIC score.

In [62]:
def forecast_drug_sales(df, forecast_months=6):
    """
    Generate forecasts for each drug and organize them by month
    """
    # Convert Date to datetime
    df['Date'] = pd.to_datetime(df['Date'])

    # Get unique drug names
    drug_names = df['Drug Name'].unique()

    # Create DataFrame to store results
    results_df = pd.DataFrame()

    # Add original columns from input data that we want to keep
    base_columns = ['Disease Category', 'Drug Category', 'Drug Name', 'Dosage', 'Retail Price','Purchase Price', 'Sales', 'Date', 'Year', 'Quarter', 'Month','Lag_1', 'Lag_2', 'Rolling_Mean_3', 'EMA_3', 'Mean Sale', 'CV',
'Buffer Percentage', 'Buffer Stock']

    for col in base_columns:
        if col in df.columns:
            # Get the latest value for each drug
            latest_values = df.groupby('Drug Name')[col].last()
            results_df[col] = [latest_values.get(drug, '') for drug in drug_names]

    # Generate forecasts for each drug
    for drug_name in drug_names:
        print(f"Processing {drug_name}...")

        # Filter data for this drug and prepare time series
        drug_data = df[df['Drug Name'] == drug_name]
        sales_ts = drug_data.groupby('Date')['Sales'].sum()

        try:
            # Fit SARIMA model
            # Using fixed parameters for simplicity and speed
            # Can reintegrate the parameter search if needed
            model = SARIMAX(
                sales_ts,
                order=(1, 1, 1),
                seasonal_order=(1, 1, 1, 12),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            results = model.fit(disp=False)

            # Generate forecast
            forecast = results.get_forecast(steps=forecast_months)
            predictions = forecast.predicted_mean

            # Add predictions to results DataFrame
            for i, pred in enumerate(predictions):
                # Calculate forecast date
                forecast_date = sales_ts.index[-1] + pd.DateOffset(months=i+1)
                # Create column name in desired format
                col_name = f'Predicted_Sales_{forecast_date.strftime("%Y-%m")}'
                # Add prediction to results
                results_df.loc[results_df['Drug Name'] == drug_name, col_name] = round(pred, 2)

        except Exception as e:
            print(f"Error processing {drug_name}: {str(e)}")
            # Fill with NaN if forecasting fails
            for i in range(forecast_months):
                forecast_date = sales_ts.index[-1] + pd.DateOffset(months=i+1)
                col_name = f'Predicted_Sales_{forecast_date.strftime("%Y-%m")}'
                results_df.loc[results_df['Drug Name'] == drug_name, col_name] = np.nan

    # Save to Excel
    results_df.to_excel('C:/Users/ASUS/OneDrive/Desktop/sarima_drug_sales_forecasts.xlsx', index=False)

    return results_df

In [63]:
# Read your data
df = pd.read_excel('C:/Users/ASUS/OneDrive/Desktop/One_Drug_Data_Featured.xlsx')

# Generate forecasts
forecasts= forecast_drug_sales(df)

Processing AMLODAC 5MG...
Processing AMLONG 10MG...
Processing AMLONG 2. 2. 2. 2.5MG...
Processing AMLONG 5MG...
Processing AMLOPRESS 5MG...
Processing AMLOSUN 5MG...
Processing STAMLO 5MG...
Processing STAMLO 10MG...
Processing AMILODIPINE (CADLA) 10MG...
Processing ASPIN 100MG...
Processing ASPIRIN (SPC) 75MG...
Processing ASPRIN 75MG...
Processing ASTRIX 100MG...
Processing CARDIPRIN 100MG...
Processing CAREPRIN 75MG...
Processing ECORIN 150MG...
Processing ECORIN 75MG...
Processing ECOSPRIN 75MG...
Processing XSPRIN 75MG...
Processing LOSARTAN K 75MG...
Processing ASPIRIN 150MG...
Processing CLOPACT 75MG...
Processing CLOPID 75MG...
Processing CLOPIDOGREL(MEPRO) 75MG...
Processing CLOPIDOGREL 75MG...
Processing CLOPIL 75MG...
Processing CLOPILET 75MG...
Processing CLOPITAB 75MG...
Processing CLOPIVAS 75MG...
Processing CLOPIVIC 75MG...
Processing DEPLATTE 75MG...
Processing NOKLOT 75MG...
Processing PIDLET 75MG...
Processing PLAGERIN 75MG...
Processing PLATEX 75MG...
Processing PLA

In [65]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_forecast(y_true, y_pred):
    """
    Evaluate forecast accuracy using MAPE, RMSE, and MAE.
    """
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return {"MAPE": mape, "RMSE": rmse, "MAE": mae}

# Example usage (assuming we have actual sales for validation)
actual_sales = df[df['Date'] >= '2024-01-01']['Sales']  # Adjust date range
predicted_sales = forecasts.loc[:, 'Predicted_Sales_2024-03':'Predicted_Sales_2025-04'].values.flatten()

# Compute accuracy
accuracy_metrics = evaluate_forecast(actual_sales, predicted_sales)
print("Forecast Accuracy:", accuracy_metrics)


ValueError: operands could not be broadcast together with shapes (5098,) (0,) 

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def time_series_cross_validation(df, forecast_months=6):
    """
    Perform time series cross-validation.
    """
    tscv = TimeSeriesSplit(n_splits=5)

    errors = []
    for train_idx, test_idx in tscv.split(df):
        train, test = df.iloc[train_idx], df.iloc[test_idx]
        train_sales = train.groupby('Date')['Sales'].sum()
        test_sales = test.groupby('Date')['Sales'].sum()

        # Fit SARIMA model
        model = SARIMAX(train_sales, order=(1,1,1), seasonal_order=(1,1,1,12))
        results = model.fit()

        # Predict next forecast_months
        forecast = results.get_forecast(steps=len(test_sales)).predicted_mean

        # Evaluate errors
        errors.append(mean_absolute_error(test_sales, forecast))

    return np.mean(errors)

# Example usage
cv_error = time_series_cross_validation(df)
print("Time Series Cross-Validation Error (MAE):", cv_error)
